# RackSmart: final tuned algorithm (York)

In [ ]:
CROSSWALK = "location_stop_crosswalk.csv"        # location -> stop crosswalk (travel network)
STOP_DIST = "stop_distance_ft.csv"               # stop-to-stop walking distance (feet)
LANE_LABELS = "lane_labels.csv"                  # staging lane stops (stop_id); trips end at the average lane; None = round trip from dispatch
DEPOT_STOP = "D_183183.9_5933.3"                 # dispatch board stop: where every trip starts (node_type dispatch in stops.csv)
PICKS     = "combined_picks_sep2024_sep2026.csv.gz"   # pick transactions: CREATE_DT, USER_NAME, ORD_NO, FROM_LOCATION, FROM_ITEM, FROM_QTY, FROM_UOM
INVENTORY = "York_Inventory_On_Hand.csv"         # inventory on hand (LOCATION, ITEM, UOM, QTY); cases on hand set the cost of a move
CLUSTERS  = "sku_clusters.csv"                   # SKU clusters (sku, cluster); None = run without clusters
RACK_HEIGHTS = "rack_locations_pallet_capacity.csv"   # LOCATION, PALLET_HEIGHT; swaps only between faces of the same height; None = no height rule
CASES_PER_PALLET = "cases_per_pallet.csv"        # item, cases_per_pallet (for refill trips); None = refills off
REFILL_ORIGIN    = "refill_origin.csv"           # item, origin_location (where its refills start); optional
RESLOT = "2025-11"      # a month when the layout was reset: the look-back never reaches back past it; None if there was none
PLAN_MONTHS = None      # e.g. ["2026-06"]; None = every month from Jan 2026 on
WAGE, MPH, COST_PER_CASE = 45.50, 1.25, 0.17   # picker wage $/hr, walking speed (mph), $ to move one case
FORKLIFT_MPH = 3.0          # refill truck speed (mph)
REFILLS_PER_PALLET = 1.49   # refills per pallet-load of cases picked
SETTINGS = dict(lookback=1, K=30, margin=2.0, top=20, passes=2, cap=100, cluster=3.0, refill=True)   # the tuned values

## 1. Engine

In [ ]:
from dataclasses import dataclass, field
import numpy as np

@dataclass
class Inputs:
    name: str
    D: np.ndarray             # (n+1) x (n+1) walking distances between pick locations; last row/col = dispatch point (depot)
    depot: int                # index of the dispatch point (= n)
    cases: list               # cases held per location; drives move cost
    golden: list              # location ids eligible as swap partners
    dollars_per_unit: float   # $ of picker time per distance unit walked
    cost_per_case: float      # $ to move one case
    tours_by_period: dict     # period label -> list of tours; a tour = list of location ids in visit order
    unit_note: str = ""
    extra: dict = field(default_factory=dict)

In [ ]:
import os
import numpy as np
from collections import defaultdict

class World:
    def __init__(self, D, depot, cases, golden, dollars_per_unit, cost_per_case):
        self.D = [list(map(float, r)) for r in np.asarray(D)]
        self.depot = depot
        self.cases = cases
        self.golden = list(golden)
        self.dpu = dollars_per_unit
        self.cpc = cost_per_case
        self.n = len(cases)

    def tour_len(self, seq):
        if not seq: return 0.0
        D, dep = self.D, self.depot
        s = D[dep][seq[0]] + D[seq[-1]][dep]
        for a, b in zip(seq, seq[1:]): s += D[a][b]
        return s

    def replan(self, seq, m):
        if not any(l in m for l in seq): return seq
        D, dep = self.D, self.depot
        cur = [l for l in seq if l not in m]
        for l in seq:
            if l in m:
                x = m[l]
                if x in cur: continue
                best_c, best_i, prev = None, 0, dep
                for i in range(len(cur) + 1):
                    nxt = cur[i] if i < len(cur) else dep
                    c = D[prev][x] + D[x][nxt] - D[prev][nxt]
                    if best_c is None or c < best_c: best_c, best_i = c, i
                    prev = nxt
                cur = cur[:best_i] + [x] + cur[best_i:]
        return cur

    def detour_ranking(self, tours):
        D, dep = self.D, self.depot
        det, vis = defaultdict(float), defaultdict(int)
        for t in tours:
            s = [dep] + list(t) + [dep]
            for i in range(1, len(s) - 1):
                x = D[s[i-1]][s[i]] + D[s[i]][s[i+1]] - D[s[i-1]][s[i+1]]
                det[s[i]] += max(x, 0.0); vis[s[i]] += 1
        order = sorted(det, key=lambda l: -det[l])
        return order, dict(vis), dict(det)

class Planner:
    def __init__(self, W, train, K=5, cap=150):
        self.W, self.train, self.K, self.cap = W, train, K, cap
        self.order_detour, self.vis, self.det = W.detour_ranking(train)
        self.through = defaultdict(list)
        for i, t in enumerate(train):
            for l in set(t): self.through[l].append(i)
        self.dep_dist = [W.D[W.depot][l] for l in range(W.n)]

    def cases(self, l): return self.W.cases[l]

    def quietest_golden(self, hot, moved):
        if hot in self.gset: return []
        c = [l for l in self.W.golden if l not in moved and (os.environ.get('VISITED_ONLY','1') == '0' or l in self.vis)]
        c.sort(key=lambda l: (self.vis.get(l, 0), self.cases(l), self.dep_dist[l]))
        return c

    @property
    def gset(self):
        if not hasattr(self, "_g"): self._g = set(self.W.golden)
        return self._g

    def try_swap(self, hot, partner, moved, cur_len):
        W = self.W
        touched = set(self.through.get(hot, [])) | set(self.through.get(partner, []))
        w = {**moved, hot: partner, partner: hot}
        before = sum(cur_len[i] for i in touched)
        after_lens = {i: W.tour_len(W.replan(self.train[i], w)) for i in touched}
        saved = (before - sum(after_lens.values())) * W.dpu
        cost = W.cpc * (self.cases(hot) + self.cases(partner))
        return saved, cost, w, after_lens

    def greedy(self, order, cost_test=True, n_force=None, partners=None, moved0=None):
        W = self.W
        partners = partners or self.quietest_golden
        moved = dict(moved0 or {}); cur_len = [W.tour_len(W.replan(t, moved)) if moved else W.tour_len(t) for t in self.train]
        n, tcost = 0, 0.0
        limit = self.cap if cost_test else n_force
        for hot in order:
            if n >= limit: break
            if hot in moved: continue
            parts = partners(hot, moved)[: (self.K if cost_test else 1)]
            best = None
            for p in parts:
                saved, cost, w, al = self.try_swap(hot, p, moved, cur_len)
                if cost_test:
                    if saved >= cost * getattr(self, 'margin', 1.0) and (best is None or saved - cost > best[0] - best[1]): best = (saved, cost, w, al)
                else:
                    best = (saved, cost, w, al)
            if best:
                _, cost, moved, al = best
                for i, v in al.items(): cur_len[i] = v
                n += 1; tcost += cost
        return moved, n, tcost

    def cluster_step(self, moved0, loc_cluster, order):
        vis_in = defaultdict(list)
        for l, c in loc_cluster.items(): vis_in[c].append(self.vis.get(l, 0))
        ranked = sorted(vis_in, key=lambda c: -np.mean(vis_in[c]))
        pool = sorted(loc_cluster, key=lambda l: self.dep_dist[l]); block, pos = {}, 0
        for c in ranked:
            block[c] = set(pool[pos:pos + len(vis_in[c])]); pos += len(vis_in[c])
        mis = [l for l in order if l in loc_cluster and l not in block[loc_cluster[l]]]
        def own_block_spots(hot, moved):
            c = loc_cluster[hot]
            return sorted((l for l in block[c] if loc_cluster.get(l) != c and l not in moved), key=lambda l: (self.vis.get(l, 0), self.cases(l)))
        return self.greedy(mis, True, partners=own_block_spots, moved0=moved0)

    def plan(self, policy, n_ref=None):
        if policy == "detour+cost":
            return self.greedy(self.order_detour, True)
        if policy == "velocity+cost":
            order = sorted(self.vis, key=lambda l: -self.vis[l])
            return self.greedy(order, True)
        if policy == "classic_abc":
            order = sorted(self.vis, key=lambda l: -self.vis[l])
            return self.greedy(order, False, n_force=n_ref)
        raise ValueError(policy)

def replay(W, test, moved):
    tot_before = sum(W.tour_len(t) for t in test)
    if not moved: return 0.0, tot_before, 0.0
    saved = 0.0
    for t in test:
        if any(l in moved for l in t):
            saved += W.tour_len(t) - W.tour_len(W.replan(t, moved))
    return saved, tot_before, saved / tot_before if tot_before else 0.0

def move_cost(W, moved):
    seen, c = set(), 0.0
    for a, b in moved.items():
        if a in seen: continue
        seen.add(a); seen.add(b); c += W.cpc * (W.cases[a] + W.cases[b])
    return c

In [ ]:
from collections import defaultdict
import numpy as np
import re

class PlannerPlus(Planner):
    def __init__(self, W, train, **kw):
        super().__init__(W, train, **kw)

    def try_swap(self, hot, partner, moved, cur_len):
        saved, cost, w, al = super().try_swap(hot, partner, moved, cur_len)
        rf = getattr(self, "refill", None)
        if rf:
            if rf[0] == "prod":
                _, rate, orow, ratio = rf; d = 0.0
                for a, b in ((hot, partner), (partner, hot)):
                    if a in orow: d += rate[a] * (orow[a][b] - orow[a][a])
                cost += 2 * self.W.dpu * ratio * d
            else:
                rate, row, ratio = rf
                cost += 2 * self.W.dpu * ratio * (rate[hot] * (row[partner] - row[hot]) + rate[partner] * (row[hot] - row[partner]))
        cr = getattr(self, "crowd", None)
        if cr:
            aisle, kappa = cr
            def loads(mv):
                ld = {}
                for l, v in self.vis.items():
                    a = aisle[mv.get(l, l)]; ld[a] = ld.get(a, 0) + v
                return ld
            la, lb = loads(moved), loads(w)
            cost += kappa * (sum(v * v for v in lb.values()) - sum(v * v for v in la.values()))
        return saved, cost, w, al
    def nearer_quiet(self, hot, moved):
        dh, vh = self.dep_dist[hot], self.vis.get(hot, 0)
        sc = []
        for l, v in self.vis.items():
            if l == hot or l in moved: continue
            dd, dv = dh - self.dep_dist[l], vh - v
            if dd > 0 and dv > 0: sc.append((dd * dv, l))
        sc.sort(reverse=True)
        return [l for _, l in sc]

    def plus(self, order, K=10, passes=3, ring=True):
        old = self.K; self.K = K
        moved, n, c = self.greedy(order, True)
        for _ in range(passes - 1 if ring else 0):
            m2, n2, c2 = self.greedy(order, True, partners=self.nearer_quiet, moved0=moved)
            if n2 == 0 and m2 == moved: break
            moved, n, c = m2, n + n2 - 0, c + c2
        self.K = old
        return moved, n, c

    def cluster_two_way(self, moved0, loc_cluster, order):
        vis_in = defaultdict(list)
        for l, c in loc_cluster.items(): vis_in[c].append(self.vis.get(l, 0))
        ranked = sorted(vis_in, key=lambda c: -np.mean(vis_in[c]))
        pool = sorted(loc_cluster, key=lambda l: self.dep_dist[l]); band, pos = {}, 0
        for c in ranked:
            for l in pool[pos:pos + len(vis_in[c])]: band[l] = c
            pos += len(vis_in[c])
        mis = [l for l in order if l in loc_cluster and band[l] != loc_cluster[l]]
        def match(hot, moved):
            tgt, here = loc_cluster[hot], band[hot]
            both = [l for l in loc_cluster if l not in moved and band[l] == tgt and loc_cluster[l] == here]
            one = [l for l in loc_cluster if l not in moved and band[l] == tgt and loc_cluster[l] != tgt and loc_cluster[l] != here]
            f = lambda l: (self.vis.get(l, 0), self.cases(l))
            return sorted(both, key=f) + sorted(one, key=f)
        return self.greedy(mis, True, partners=match, moved0=moved0)

    def rs_plus(self): return self.plus(self.order_detour, K=10, passes=3)

    def build_affinity(self, top=20, min_count=2):
        from itertools import combinations
        from collections import Counter
        cnt = Counter()
        for t in self.train:
            u = sorted(set(t))
            if len(u) > 40: u = u[:40]
            for a, b in combinations(u, 2): cnt[(a, b)] += 1
        nb = defaultdict(list)
        for (a, b), c in cnt.items():
            if c >= min_count: nb[a].append((c, b)); nb[b].append((c, a))
        self.aff = {a: sorted(v, reverse=True)[:top] for a, v in nb.items()}
        self.Dn = np.asarray(self.W.D)

        self.pull = {a: sum(c * self.Dn[a][b] for c, b in v) for a, v in self.aff.items()}

    def aff_partners(self, hot, moved):
        v = self.aff.get(hot)
        if not v: return []
        idx = np.array([b for _, b in v]); w = np.array([c for c, _ in v], float)
        score = self.Dn[:, idx] @ w
        vh = self.vis.get(hot, 0)
        cand = [l for l in np.argsort(score)[:400] if l != hot and l not in moved and 0 < self.vis.get(l, 0) < vh]
        return [int(l) for l in cand]

    def affinity_v2(self, K=10, passes=2):
        self.build_affinity(); order = sorted(self.pull, key=lambda a: -self.pull[a])
        old = self.K; self.K = K; moved, n, c = self.greedy(order, True, partners=self.aff_partners)
        for _ in range(passes - 1):
            m2, n2, c2 = self.greedy(order, True, partners=self.aff_partners, moved0=moved)
            if m2 == moved: break
            moved, n, c = m2, n + n2, c + c2
        self.K = old; return moved, n, c

    def family_grouping(self, n_swaps, min_jac=0.10, fam_size=8, cost_test=False):
        from itertools import combinations
        from collections import Counter
        cnt = Counter()
        for t in self.train:
            u = sorted(set(t))[:40]
            for a, b in combinations(u, 2): cnt[(a, b)] += 1
        nb = defaultdict(list)
        for (a, b), c in cnt.items():
            j = c / (self.vis.get(a, 0) + self.vis.get(b, 0) - c)
            if j >= min_jac: nb[a].append((j, b)); nb[b].append((j, a))
        seen, fams = set(), []
        for s in sorted(self.vis, key=lambda l: -self.vis[l]):
            if s in seen: continue
            f = [s]; seen.add(s)
            for j, b in sorted(nb.get(s, []), reverse=True):
                if len(f) >= fam_size: break
                if b not in seen and b in self.vis: f.append(b); seen.add(b)
            fams.append(f)
        fams.sort(key=lambda f: -sum(self.vis[l] for l in f))
        slots = sorted(range(self.W.n), key=lambda l: self.dep_dist[l]); target, k = {}, 0
        for f in fams:
            for l in sorted(f, key=lambda l: -self.vis[l]):
                target[l] = slots[k]; k += 1
        order = [l for f in fams for l in sorted(f, key=lambda l: -self.vis[l])]
        parts = lambda hot, moved: [target[hot]] if target[hot] != hot and target[hot] not in moved else []
        old = self.K; moved, n, c = self.greedy(order, cost_test, n_force=n_swaps, partners=parts); self.K = old
        return moved, n, c

def limit_swaps(W, tours, moved, cap):
    keep, seen = {}, set()
    for a, b in moved.items():
        if a in seen: continue
        if cap and len(seen) // 2 >= cap: break
        seen.add(a); seen.add(b); keep[a] = b; keep[b] = a
    return keep

In [ ]:
import numpy as np

MAX_CLUSTER = 20

class _ClusterPlanner(PlannerPlus):
    weight = 0.0; mates = None

    def set_clusters(self, lc, weight):
        self.weight = weight; by = {}
        for l, c in lc.items(): by.setdefault(c, []).append(l)
        self.mates = {l: sorted([x for x in by[c] if x != l], key=lambda x: -self.vis.get(x, 0))[:MAX_CLUSTER] for l, c in lc.items()}
        counts = [c for v in self.aff.values() for c, _ in v]
        self.typical = float(np.median(counts)) if counts else 1.0

    def aff_partners(self, hot, moved):
        v = list(self.aff.get(hot, []))
        if self.weight and self.mates and hot in self.mates:
            have = {b for _, b in v}
            v += [(self.weight * self.typical, b) for b in self.mates[hot] if b not in have]
        if not v: return []
        idx = np.array([b for _, b in v]); w = np.array([c for c, _ in v], float)
        score = self.Dn[:, idx] @ w
        vh = self.vis.get(hot, 0)
        cand = [l for l in np.argsort(score)[:400] if l != hot and l not in moved and 0 < self.vis.get(l, 0) < vh]
        return [int(l) for l in cand]

def plan(tours, D, depot, cases, golden, dollars_per_unit, cost_per_case, groups=None, loc_cluster=None, refill=None, window_months=1, settings=None):
    p = dict(SETTINGS, **(settings or {}))
    W = World(D, depot, cases, golden, dollars_per_unit / max(window_months, 1), cost_per_case)
    if loc_cluster and p["cluster"]:
        pl = _ClusterPlanner(W, tours); pl.build_affinity(top=p["top"]); pl.set_clusters(loc_cluster, p["cluster"])
    else:
        pl = PlannerPlus(W, tours); pl.build_affinity(top=p["top"])
    pl.K, pl.margin = p["K"], p["margin"]
    if refill and p["refill"]: pl.refill = refill
    order = [l for l in pl.order_detour if l in pl.aff]
    same = (lambda hot, q: groups[q] == groups[hot]) if groups is not None else (lambda hot, q: True)
    parts = lambda hot, mv: [q for q in pl.aff_partners(hot, mv) if same(hot, q)]
    moved, _, _ = pl.greedy(order, True, partners=parts)
    for _ in range(p["passes"] - 1):
        m2, _, _ = pl.greedy(order, True, partners=parts, moved0=moved)
        if m2 == moved: break
        moved = m2
    return limit_swaps(W, tours, moved, p["cap"])

## 2. Load data, distances, tours, aisle fence, short/tall groups

In [ ]:
import os, re, numpy as np, pandas as pd
from collections import Counter
from itertools import combinations

def load_york():
    cw = pd.read_csv(CROSSWALK); cw = cw[(cw.status == "ok") & cw.stop_id.notna() & (cw.level == "A")]
    sd = pd.read_csv(STOP_DIST, index_col=0)
    stops = list(sd.index); sidx = {s: i for i, s in enumerate(stops)}
    dock_stop = DEPOT_STOP
    cols = ["CREATE_DT", "USER_NAME", "FROM_ITEM", "FROM_LOCATION", "FROM_QTY", "FROM_UOM", "ORD_NO"]
    pk = pd.read_csv(PICKS, usecols=cols, dtype={c: str for c in cols if c != "FROM_QTY"})
    pk = pk[pk.FROM_UOM == "CS"].copy()                                  # case picks only
    pk["dt"] = pd.to_datetime(pk.CREATE_DT, format="mixed", errors="coerce"); pk = pk.dropna(subset=["dt", "ORD_NO"])
    pk["loc"] = pk.FROM_LOCATION.astype(str).str.strip()
    stop_of = dict(zip(cw.location.str.strip(), cw.stop_id))
    n_all = len(pk); pk = pk[pk["loc"].isin(stop_of)].copy(); print("case picks on a mapped pick face:", len(pk), "of", n_all)
    faces = sorted(pk["loc"].unique()); idx = {l: i for i, l in enumerate(faces)}; n = len(faces)
    si = np.array([sidx[stop_of[l]] for l in faces]); Ds = sd.values.astype(float)
    D = np.zeros((n + 1, n + 1)); D[:n, :n] = Ds[np.ix_(si, si)]
    D[n, :n] = Ds[sidx[dock_stop], si]                                   # trips start at the dispatch point
    if LANE_LABELS:                                                       # trips end at a staging lane: the average of all lanes (order -> lane is not in the data)
        lanes = [sidx[s] for s in pd.read_csv(LANE_LABELS).stop_id.unique()]; D[:n, n] = np.round(Ds[np.ix_(si, lanes)].mean(axis=1), 3)
    else: D[:n, n] = Ds[si, sidx[dock_stop]]                              # no lane file: round trip from dispatch
    inv = pd.read_csv(INVENTORY); inv = inv[inv.UOM == "CS"]
    cases_by = inv.assign(l=inv.LOCATION.astype(str).str.strip()).groupby("l").QTY.sum()
    typical = float(np.median(cases_by.values)); cases = [float(cases_by.get(l, typical)) for l in faces]   # missing = median location
    golden = [int(i) for i in np.argsort(D[n, :n])[: int(round(0.108 * n))]]     # nearest 10.8% of faces to dispatch (engine input; RackSmart picks partners by co-picks, not from this list)
    pk["li"] = pk["loc"].map(idx); pk["month"] = pk.dt.dt.to_period("M").astype(str)
    pk["FROM_QTY"] = pd.to_numeric(pk.FROM_QTY, errors="coerce").fillna(0)
    sku = pk.groupby(["month", "li", "FROM_ITEM"]).agg(picks=("FROM_QTY", "size"), cases=("FROM_QTY", "sum")).reset_index()
    pk = pk.sort_values(["ORD_NO", "USER_NAME", "dt"]); by = {}
    for (o, u), g in pk.groupby(["ORD_NO", "USER_NAME"], sort=False):    # a tour = one operator's picks for one order (>= 3 stops)
        seq = [g.li.iloc[0]]
        for l in g.li.iloc[1:]:
            if l != seq[-1]: seq.append(l)
        if len(seq) >= 3: by.setdefault(g.month.iloc[0], []).append((g.dt.iloc[0], seq))
    by = {m: [s for _, s in sorted(v, key=lambda x: x[0])] for m, v in by.items()}
    by = {m: v for m, v in by.items() if m not in (min(by), max(by))}      # drop the first and last (partial) months
    dpu = (WAGE / 3600.0) / (MPH * 5280 / 3600.0)                          # $ per foot walked
    print({m: len(v) for m, v in sorted(by.items())})
    groups = [1 if re.match(r"[A-Z]+", f).group(0) in ("GG", "HH") else 0 for f in faces]     # aisle fence
    if RACK_HEIGHTS:                                                       # short/tall: faces with no height on file only swap with each other
        h = pd.read_csv(RACK_HEIGHTS); h["LOCATION"] = h.LOCATION.astype(str).str.strip()
        code = {r.LOCATION: {"SHORT": 1, "TALL": 2}.get(r.PALLET_HEIGHT, 0) for r in h.itertuples()}
        groups = [g + 2 * code.get(f, 0) for g, f in zip(groups, faces)]
    return Inputs("york", D, n, cases, golden, dpu, COST_PER_CASE, by, unit_note="feet",
                  extra=dict(faces=faces, groups=groups, sku=sku, stops=(cw, sd, sidx, stop_of, [stop_of[l] for l in faces])))
INP = load_york()

def location_clusters(inp, months, train, max_size=20):
    """Each face takes the cluster of the item picked there most in the look-back months; clusters over 20 faces are split by co-pick pairs."""
    cl = pd.read_csv(CLUSTERS); sku_cluster = dict(zip(cl.sku.astype(str), cl.cluster))
    f = inp.extra["sku"]; f = f[f.month.isin(months)].assign(item=lambda d: d.FROM_ITEM.astype(str))
    top = f.groupby(["li", "item"]).cases.sum().reset_index().sort_values("cases", ascending=False).drop_duplicates("li")
    lc = {int(l): sku_cluster[i] for l, i in zip(top.li, top["item"]) if i in sku_cluster}
    members = {}
    for l, c in lc.items(): members.setdefault(c, []).append(l)
    big = [c for c, m in members.items() if len(m) > max_size]
    if big:
        visits, both = Counter(), Counter()
        for t in train:
            u = list(dict.fromkeys(t))[:40]; visits.update(u); both.update(combinations(sorted(u), 2))
        rows = [(c / (visits[a] + visits[b] - c), c, a, b) for (a, b), c in both.items() if c >= 2]
        for c in big:
            s = set(members[c]); sub = [r for r in rows if r[2] in s and r[3] in s]
            fams = build_families(sub, 0, max_size)
            for k, fam in enumerate(fams):
                for l in fam: lc[l] = f"{c}.{k}"
            for l in s:
                if not any(l in fam for fam in fams): lc[l] = f"{c}.rest"
    return lc

def build_families(rows, cutoff_pct, fam_cap=8):
    """Pairs scoring at or above the cutoff percentile are merged into families, best pair first, never above fam_cap members."""
    if not rows: return []
    cut = np.percentile([r[0] for r in rows], cutoff_pct)
    fam, members, next_id = {}, {}, 0
    for s, c, a, b in sorted(rows, key=lambda r: (-r[0], -r[1], r[2], r[3])):
        if s < cut: break
        fa, fb = fam.get(a), fam.get(b)
        if fa is None and fb is None:
            i = next_id; next_id += 1; members[i] = [a, b]; fam[a] = fam[b] = i
        elif fa is not None and fb is None and len(members[fa]) < fam_cap:
            members[fa].append(b); fam[b] = fa
        elif fb is not None and fa is None and len(members[fb]) < fam_cap:
            members[fb].append(a); fam[a] = fb
        elif fa is not None and fb is not None and fa != fb and len(members[fa]) + len(members[fb]) <= fam_cap:
            for x in members[fb]: fam[x] = fa
            members[fa] += members[fb]; del members[fb]
    return list(members.values())

def refill_inputs(inp, months, per_pallet=None):
    """Per face: refills a month (from demand) and the distance row from the item's bulk storage. The item at a face is the one picked there most."""
    if not (SETTINGS["refill"] and CASES_PER_PALLET): return None
    per_pallet = per_pallet or REFILLS_PER_PALLET
    cw, sd, sidx, stop_of, face_stops = inp.extra["stops"]
    allw = pd.read_csv(CROSSWALK); allw = allw[(allw.status == "ok") & allw.stop_id.notna()].drop_duplicates(allw.columns[allw.columns.get_loc("location")])
    any_level = dict(zip(allw.location.astype(str).str.strip(), allw.stop_id))      # bulk storage can be on any level
    cpp = pd.read_csv(CASES_PER_PALLET, dtype={"item": str}).set_index("item").cases_per_pallet.to_dict()
    origin, everything = {}, {}
    if REFILL_ORIGIN:                                  # where each item's refills actually started, with the number of refills
        o = pd.read_csv(REFILL_ORIGIN, dtype={"item": str, "origin_location": str}); o["stop"] = o.origin_location.map(any_level); o = o.dropna(subset=["stop"])
        for i, st, n in zip(o["item"], o.stop, o.refills): origin.setdefault(i, {})[st] = origin.get(i, {}).get(st, 0) + n; everything[st] = everything.get(st, 0) + n
    rows = {}
    def row(stop):
        if stop not in rows: rows[stop] = sd.loc[stop, face_stops].values.astype(float)
        return rows[stop]
    avg_rows = {}
    def avg_row(key, w):                                # average distance from an item's refill origins, weighted by refills
        if key not in avg_rows: avg_rows[key] = sum(n * row(st) for st, n in w.items()) / float(sum(w.values()))
        return avg_rows[key]
    f = inp.extra["sku"]; f = f[f.month.isin(months)].assign(item=lambda d: d.FROM_ITEM.astype(str))
    top = f.groupby(["li", "item"]).cases.sum().reset_index().sort_values("cases", ascending=False).drop_duplicates("li")
    rate, orow = np.zeros(len(inp.cases)), {}
    for li, item, cs in zip(top.li, top["item"], top.cases):
        c = cpp.get(item, np.nan)
        if not c or c != c or not everything: continue
        rate[int(li)] = per_pallet * cs / c; orow[int(li)] = avg_row(item, origin[item]) if item in origin else avg_row("__all__", everything)   # no refill history: average of all refill origins
    return ("prod", rate, orow, MPH / FORKLIFT_MPH)      # a refill trip is priced at the walking rate x (walking speed / truck speed)

## 3. Clusters and refill inputs, then plan each month and check it on the next month

In [ ]:
def plan_month(inp, month):
    """The swap list for `month`, planned from the look-back months before it. Returns {location: partner} (both directions)."""
    P = sorted(inp.tours_by_period)
    usable = [m for m in P if m < month and not (RESLOT and month > RESLOT and m <= RESLOT)]
    months = usable[-SETTINGS["lookback"]:]
    if not months: return None, []
    train = [t for m in months for t in inp.tours_by_period[m]]
    lc = location_clusters(inp, months, train) if (CLUSTERS and SETTINGS["cluster"]) else None
    moved = plan(train, inp.D, inp.depot, inp.cases, inp.golden, inp.dollars_per_unit, inp.cost_per_case, groups=inp.extra["groups"],
                 loc_cluster=lc, refill=refill_inputs(inp, months), window_months=len(months))
    return moved, months

def swap_list(inp, moved):
    faces = inp.extra["faces"]; seen = set(); rows = []
    for a, b in moved.items():
        if (b, a) in seen: continue
        seen.add((a, b)); rows.append(dict(location_a=faces[a], location_b=faces[b], cases_a=inp.cases[a], cases_b=inp.cases[b]))
    return pd.DataFrame(rows)

def check_month(inp, month, moved):
    """Replay the swaps on the month's real tours: walking saved, cost of the moves, net (refill driving not included)."""
    We = World(inp.D, inp.depot, inp.cases, inp.golden, inp.dollars_per_unit, inp.cost_per_case)
    s, t, p = replay(We, inp.tours_by_period[month], moved); c = move_cost(We, moved)
    return dict(month=month, swaps=len(moved) // 2, walking_cut_pct=round(100 * p, 2), gross=round(s * We.dpu), move_cost=round(c), net=round(s * We.dpu - c))

months = PLAN_MONTHS or [m for m in sorted(INP.tours_by_period) if m >= "2026-01"]
plans, rows = {}, []
for m in months:
    mv, used = plan_month(INP, m)
    if mv is None: continue
    plans[m] = mv; rows.append(check_month(INP, m, mv)); print("done", m, "| planned from", used, flush=True)
res = pd.DataFrame(rows); res